In [27]:
import os
import re
import glob
import numpy as np
from scipy.optimize import least_squares
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Global cache: avoid re-loading .npz on every residual call
# ------------------------------------------------------------
_flat_cache = {}

def load_flatpack(k_index, flat_dir="lambda_singlet"):
    """
    Load and cache flat-band data for a given k_index.
    Expects files named flat_kXXXXX.npz inside flat_dir.
    """
    k_index = int(k_index)
    key = (flat_dir, k_index)
    if key in _flat_cache:
        return _flat_cache[key]

    path = os.path.join(flat_dir, f"flat_k{k_index:05d}.npz")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    z = np.load(path, allow_pickle=True)
    pack = {
        "L0flat":     z["L0flat"],          # shape: (3, N, N)
        "L1flat":     z["L1flat"],          # shape: (3, N, N)
        "Eplus_flat":  z["Eplus_flat"],     # shape: (N,)
        "Eminus_flat": z["Eminus_flat"],    # shape: (N,)
        "k":          z["k"],
        "k_index":    int(z["k_index"]),
    }
    _flat_cache[key] = pack
    return pack

# ------------------------------------------------------------
# Helper: centered subspace indices
# ------------------------------------------------------------
def _center_indices(N, subspace_dim):
    """
    Return indices for a centered subspace of size subspace_dim inside 0..N-1.
    """
    if subspace_dim > N:
        raise ValueError(f"subspace_dim={subspace_dim} > full dim N={N}")
    start = (N - subspace_dim) // 2
    end   = start + subspace_dim
    return np.arange(start, end, dtype=int)

# ------------------------------------------------------------
# Flat-band BdG Hamiltonian (restricted subspace)
# ------------------------------------------------------------
def flat_BdG(pack, *, DeltaLayer0, DeltaLayer1, mu,
             subspace_dim=10):
    """
    Build flat-band BdG Hamiltonian in a centered subspace of size subspace_dim.
    Δ0, Δ1 are length-3 complex arrays (one component per Kekulé irrep).
    """
    Δ0 = np.asarray(DeltaLayer0, complex).ravel()
    Δ1 = np.asarray(DeltaLayer1, complex).ravel()
    if Δ0.size != 3 or Δ1.size != 3:
        raise ValueError("DeltaLayer0/1 must each have length 3.")

    Eplus  = np.asarray(pack["Eplus_flat"],  float)
    Eminus = np.asarray(pack["Eminus_flat"], float)
    L0flat = np.asarray(pack["L0flat"])   # (3, N, N)
    L1flat = np.asarray(pack["L1flat"])   # (3, N, N)

    N = Eplus.shape[0]
    idx = _center_indices(N, subspace_dim)

    # restrict to subspace
    Eplus_sub  = Eplus[idx]
    Eminus_sub = Eminus[idx]
    L0_sub = L0flat[:, idx][:, :, idx]   # (3, subspace_dim, subspace_dim)
    L1_sub = L1flat[:, idx][:, :, idx]   # (3, subspace_dim, subspace_dim)

    Fp = np.diag(Eplus_sub)
    Fm = -np.diag(Eminus_sub)

    # pairing: here only sym = 0 component used; Lflat is mathematical Λ
    Gap_flat = Δ0[0] * np.conj(L0_sub[0]) + Δ1[0] * np.conj(L1_sub[0])

    I = np.eye(Fp.shape[0], dtype=complex)
    HLL = np.block([
        [Fp - mu * I,          -Gap_flat        ],
        [-Gap_flat.conj().T,    Fm + mu * I     ]
    ]).astype(complex)

    return HLL, Gap_flat

# ------------------------------------------------------------
# Lambda in flat subspace
# ------------------------------------------------------------
def Lambda_Flat(pack_flat, *, layer=0, sym=0, subspace_dim=10):
    """
    Return Λ (conjugated) in the same centered subspace as flat_BdG.
    Mathematical Λ = Lflat[sym]. Here we return conj(Λ) so that
    np.vdot(Lambda, F) = tr(Λ^T F).
    """
    Lflat_all = np.asarray(pack_flat["L0flat" if layer == 0 else "L1flat"])
    Lfull_math = Lflat_all[sym]              # Λ_math, shape (N, N)
    Lambda_full = np.conj(Lfull_math)        # Λ_code = conj(Λ_math)

    N = Lambda_full.shape[0]
    idx = _center_indices(N, subspace_dim)
    Lambda = Lambda_full[idx][:, idx]        # (subspace_dim, subspace_dim)
    return Lambda

# ------------------------------------------------------------
# Anomalous correlator F
# ------------------------------------------------------------
def Anomalous(pack, *, DeltaLayer0, DeltaLayer1, mu, T=1e-3,
              subspace_dim=10):
    """
    Compute anomalous correlator F = G_{12} in the truncated subspace.
    Uses Matsubara-like tanh kernel: t = -1/2 tanh(βE/2).
    """
    HLL, _  = flat_BdG(pack, DeltaLayer0=DeltaLayer0,
                      DeltaLayer1=DeltaLayer1,
                      mu=mu, subspace_dim=subspace_dim)
    E, U = np.linalg.eigh(HLL)

    beta = 1.0 / float(T)
    x = 0.5 * beta * np.asarray(E)
    t = -0.5 * np.tanh(x)
    G = (U * t) @ U.conj().T
    n = HLL.shape[0] // 2
    F = -G[:n, n:]   # particle-hole block
    return F, (E, U)

# ------------------------------------------------------------
# Stable Fermi function using tanh (no overflow)
# ------------------------------------------------------------
def fermi_occupation(E, T):
    """
    f(E) = 1/(1 + exp(βE)) = 0.5 * (1 - tanh(βE/2)), stable for large βE.
    """
    beta = 1.0 / float(T)
    x = 0.5 * beta * np.asarray(E)
    return 0.5 * (1.0 - np.tanh(x))

# ------------------------------------------------------------
# Brillouin-zone integration: gap contractions + density
# ------------------------------------------------------------
def IntegrateBZ_acc(*, DeltaLayer0, DeltaLayer1, mu,
                    sym=0, T=1e-3,
                    flat_dir="lambda_singlet",
                    average=True,
                    subspace_dim=10):
    """
    Loop over all k-points in flat_dir and accumulate:
        acc0 = <Λ0, F>
        acc1 = <Λ1, F>
        n    = particle density
    Everything is done in one BZ sweep using the same (E, U).
    """
    files = sorted(
        fn for fn in os.listdir(flat_dir)
        if re.match(r"flat_k\d{5}\.npz$", fn)
    )
    if not files:
        raise FileNotFoundError(f"No flat_kXXXXX.npz files found in {flat_dir}")

    acc0 = 0j
    acc1 = 0j
    total_n = 0.0
    Nk = 0

    for fn in files:
        k_index = int(fn[6:11])
        pack = load_flatpack(k_index, flat_dir=flat_dir)

        # anomalous block + spectrum
        F, (E, U) = Anomalous(
            pack,
            DeltaLayer0=DeltaLayer0,
            DeltaLayer1=DeltaLayer1,
            mu=mu, T=T,
            subspace_dim=subspace_dim
        )

        # gap contractions: Gamma = conj(Λ_math)
        Gamma0 = Lambda_Flat(pack, layer=0, sym=sym, subspace_dim=subspace_dim)
        Gamma1 = Lambda_Flat(pack, layer=1, sym=sym, subspace_dim=subspace_dim)
        # np.vdot(Gamma, F) = sum conj(Gamma)*F = sum Λ_math * F = tr(Λ_math^T F)
        acc0 += np.vdot(Gamma0, F)
        acc1 += np.vdot(Gamma1, F)

        # density from same E, U
        fE = fermi_occupation(E, T)
        G_occ = (U * fE) @ U.conj().T
        n_sub = F.shape[0]          # particle-block dimension in subspace
        n_k = float(np.trace(G_occ[:n_sub, :n_sub]).real)
        total_n += n_k

        Nk += 1

    if average and Nk:
        acc0 /= Nk
        acc1 /= Nk
        total_n /= Nk

    return acc0, acc1, total_n

# ------------------------------------------------------------
# ParticleMomentum (if you still want it separately)
# ------------------------------------------------------------
def ParticleMomentum(pack, *, DeltaLayer0, DeltaLayer1,
                     mu=0.0, T=1e-3, subspace_dim=10):
    """
    Compute particle-block correlator G_11 at a single k.
    """
    HLL, _ = flat_BdG(pack, DeltaLayer0=DeltaLayer0,
                      DeltaLayer1=DeltaLayer1,
                      mu=mu, subspace_dim=subspace_dim)
    E, U = np.linalg.eigh(HLL)
    fE = fermi_occupation(E, T)
    G = (U * fE) @ U.conj().T
    n = HLL.shape[0] // 2
    Gfirst = G[:n, :n]
    return Gfirst, (E, U)

# ------------------------------------------------------------
# Particle number over BZ as a wrapper
# ------------------------------------------------------------
def particle_number_BZ(*, DeltaLayer0, DeltaLayer1, mu=0.0, T=1e-3,
                       flat_dir="lambda_singlet", average=True,
                       subspace_dim=10):
    """
    Wrapper using IntegrateBZ_acc to get density only.
    """
    _, _, n = IntegrateBZ_acc(
        DeltaLayer0=DeltaLayer0,
        DeltaLayer1=DeltaLayer1,
        mu=mu,
        sym=0,
        T=T,
        flat_dir=flat_dir,
        average=average,
        subspace_dim=subspace_dim
    )
    return n

# ------------------------------------------------------------
# Solve gap + density at fixed V
# ------------------------------------------------------------
def solve_gap_density_free(*, sym=0, V=-1.0, T=1e-3, mu_init=0.0,
                           flat_dir="lambda_singlet",
                           x0=(5e-2, 0.0, -5e-2, 0.0),  # (ReΔ0, ImΔ0, ReΔ1, ImΔ1)
                           n_target=3.0/8.0,
                           max_nfev=800, verbose=True,
                           subspace_dim=10):
    """
    Solve for (Δ0, Δ1, μ) such that:
      Δ0 = V * acc0, Δ1 = V * acc1, n = n_target.
    """
    x0 = np.asarray([x0[0], x0[1], x0[2], x0[3], float(mu_init)], float)

    def residuals(x):
        d0 = complex(x[0], x[1])
        d1 = complex(x[2], x[3])
        mu = float(x[4])

        Δ0_vec = np.zeros(3, complex)
        Δ1_vec = np.zeros(3, complex)
        Δ0_vec[sym] = d0
        Δ1_vec[sym] = d1

        acc0, acc1, n = IntegrateBZ_acc(
            DeltaLayer0=Δ0_vec,
            DeltaLayer1=Δ1_vec,
            mu=mu,
            sym=sym,
            T=T,
            flat_dir=flat_dir,
            average=True,
            subspace_dim=subspace_dim
        )

        # gap equations (complex -> 4 real equations)
        r0 = d0 - V * acc0
        r1 = d1 - V * acc1
        # number equation
        rn = n - n_target

        return np.array([r0.real, r0.imag, r1.real, r1.imag, rn], float)

    ls = least_squares(
        residuals, x0, method="trf",
        ftol=1e-8, xtol=1e-8, gtol=1e-8,
        max_nfev=max_nfev
    )

    d0 = complex(ls.x[0], ls.x[1])
    d1 = complex(ls.x[2], ls.x[3])
    mu = float(ls.x[4])

    if verbose:
        print(f"[free] success={ls.success} resnorm={np.linalg.norm(ls.fun):.3e} msg={ls.message}")
        print(f"Δ0={d0:+.6e}, Δ1={d1:+.6e}, μ={mu:+.6e}")

    info = {
        "success": bool(ls.success),
        "resnorm": float(np.linalg.norm(ls.fun)),
        "message": ls.message,
    }
    return (d0, d1, mu), info

import numpy as np
import matplotlib.pyplot as plt



sym = 0
V= -0.00080
flat_dir = "lambda_triplet"
#
subspace_dim=10;
n_target = 3.0/8.0+(subspace_dim/2-1)
T_start, T_end, npts = 0.00002, 0.000085, 10
#
prev = (0.00032,0, 0.00032, 0, 0.0)
max_nfev = 800
verbose = True

# --- sweep & collect ---
Ts = np.linspace(T_start, T_end, npts)
results = []

for i, T in enumerate(Ts):
    if verbose:
        print(f"\n=== [{i+1}/{len(Ts)}] T = {T:+.6f}, target n = {n_target:.8f} ===")

    (d0, d1, mu), info = solve_gap_density_free(
        sym=sym, V=V, T=T, mu_init=prev[4], flat_dir=flat_dir,
        x0=(prev[0], prev[1], prev[2], prev[3]),
        n_target=n_target, max_nfev=max_nfev, verbose=verbose,subspace_dim=subspace_dim
    )

    # update continuation seed
    prev = (d0.real, d0.imag, d1.real, d1.imag, mu)

    
    # consistency check (one BZ sweep)
    Δ0 = np.zeros(3, complex); Δ1 = np.zeros(3, complex)
    Δ0[sym] = d0; Δ1[sym] = d1
    acc0, acc1, n = IntegrateBZ_acc(
            DeltaLayer0=Δ0, DeltaLayer1=Δ1,
            mu=mu,
            sym=sym, T=T,
            flat_dir=flat_dir, average=True,
            subspace_dim=subspace_dim
        )

    results.append({
            "T": T, "Δ0": d0, "Δ1": d1, "μ": mu,
            "n": n, "resnorm": info["resnorm"], "success": info["success"],
            "Δ0_minus_Vacc0": d0 - V*acc0, "Δ1_minus_Vacc1": d1 - V*acc1
        })

    print(f"Δ0={d0:+.6e}, Δ1={d1:+.6e}, μ={mu:+.6e}, n={n:.8f}, res={info['resnorm']:.2e}")




=== [1/10] T = +0.000020, target n = 4.37500000 ===


KeyboardInterrupt: 

In [18]:
import numpy as np, os

out_dir = "fig3/V=1"; os.makedirs(out_dir, exist_ok=True)
npz_path = os.path.join(out_dir, "posthoc_results.npz")

Ts      = np.array([r["T"] for r in results], float)
d0_re   = np.array([r["Δ0"].real for r in results], float)
d0_im   = np.array([r["Δ0"].imag for r in results], float)
d1_re   = np.array([r["Δ1"].real for r in results], float)
d1_im   = np.array([r["Δ1"].imag for r in results], float)
mu_arr  = np.array([r["μ"] for r in results], float)
n_arr   = np.array([r["n"] for r in results], float)
res_arr = np.array([r["resnorm"] for r in results], float)
ok_arr  = np.array([r["success"] for r in results], bool)

d0m_re  = np.array([r["Δ0_minus_Vacc0"].real for r in results], float)
d0m_im  = np.array([r["Δ0_minus_Vacc0"].imag for r in results], float)
d1m_re  = np.array([r["Δ1_minus_Vacc1"].real for r in results], float)
d1m_im  = np.array([r["Δ1_minus_Vacc1"].imag for r in results], float)

np.savez(
    npz_path,
    Ts=Ts,
    d0_re=d0_re, d0_im=d0_im, d1_re=d1_re, d1_im=d1_im,
    mu=mu_arr, n=n_arr, resnorm=res_arr, success=ok_arr,
    d0_minus_Vacc0_re=d0m_re, d0_minus_Vacc0_im=d0m_im,
    d1_minus_Vacc1_re=d1m_re, d1_minus_Vacc1_im=d1m_im
)
print("Saved:", npz_path)

Saved: fig3/V=1/posthoc_results.npz
